# 🔍 TruthLens — Deepfake Detector Training (Colab GPU)

**Instructions:**
1. Go to `Runtime → Change runtime type → T4 GPU` before running
2. Run all cells top to bottom (`Runtime → Run all`)
3. When prompted, upload your `kaggle.json` API key
4. The trained model is saved to Google Drive at `MyDrive/TruthLens/deepfake_detector_model.keras`
5. Download it and place it in your local `TruthLens/` folder

## ✅ Step 1 — Verify GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU available: {gpus[0].name}')
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print('❌ No GPU found — go to Runtime → Change runtime type → T4 GPU')

## 📦 Step 2 — Mount Google Drive (to save the model)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
MODEL_SAVE_PATH = '/content/drive/MyDrive/TruthLens/deepfake_detector_model.keras'
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
print(f'Model will be saved to: {MODEL_SAVE_PATH}')

## 📥 Step 3 — Download Dataset from Kaggle

You need a **Kaggle API key** (`kaggle.json`).  
Get it from: https://www.kaggle.com/settings → Account → API → Create New Token

In [ ]:
from google.colab import files

print('Upload your kaggle.json file:')
uploaded = files.upload()  # select your kaggle.json

import os, shutil
os.makedirs('/root/.config/kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials set.')

In [ ]:
# Download and unzip the dataset (~2-5 min depending on Colab speed)
!pip install -q kaggle
!kaggle datasets download -d manjilkarki/deepfake-and-real-images -p /content/data --unzip
print('✅ Dataset downloaded.')

In [ ]:
# Verify dataset structure
import os
DATASET_BASE = None

# Auto-detect dataset root (handles different zip layouts)
for root, dirs, files in os.walk('/content/data'):
    if 'Train' in dirs and 'Test' in dirs:
        DATASET_BASE = root
        break

if DATASET_BASE:
    print(f'✅ Dataset root found: {DATASET_BASE}')
    for split in ['Train', 'Test', 'Validation']:
        for cls in ['Fake', 'Real']:
            p = os.path.join(DATASET_BASE, split, cls)
            if os.path.exists(p):
                n = len(os.listdir(p))
                print(f'  {split}/{cls}: {n} images')
else:
    print('❌ Could not find dataset. Check /content/data structure.')
    !find /content/data -maxdepth 4 -type d

## 🧠 Step 4 — Build & Train Model (EfficientNetB0 + Transfer Learning)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import numpy as np

# Mixed precision for ~2x GPU speedup
tf.keras.mixed_precision.set_global_policy('mixed_float16')

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32


def make_dataset(split_path, augment=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        split_path,
        labels='inferred',
        color_mode='rgb',
        seed=42,
        batch_size=BATCH_SIZE,
        image_size=IMAGE_SIZE,
    )
    if augment:
        aug = tf.keras.Sequential([
            layers.RandomFlip('horizontal'),
            layers.RandomRotation(0.1),
            layers.RandomZoom(0.1),
            layers.RandomContrast(0.2),
            layers.RandomBrightness(0.2),
        ])
        ds = ds.map(lambda x, y: (aug(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


train_data = make_dataset(os.path.join(DATASET_BASE, 'Train'), augment=True)
val_data   = make_dataset(os.path.join(DATASET_BASE, 'Validation'), augment=False)
test_data  = make_dataset(os.path.join(DATASET_BASE, 'Test'), augment=False)
print('✅ Datasets ready.')

In [ ]:
def build_model():
    backbone = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMAGE_SIZE, 3),
    )
    backbone.trainable = False  # frozen for Phase 1

    inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
    x = backbone(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid', dtype='float32')(x)

    return tf.keras.Model(inputs, outputs)


model = build_model()
model.summary()

In [ ]:
# ── Phase 1: Train classifier head (backbone frozen) ─────────────────────────
print('\n=== Phase 1: Training classifier head (backbone frozen) ===')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

callbacks_p1 = [
    EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, mode='max'),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7),
]

history_p1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks_p1,
)
print(f'Phase 1 best val_accuracy: {max(history_p1.history["val_accuracy"]):.4f}')

In [ ]:
# ── Phase 2: Fine-tune top 30 EfficientNetB0 layers ─────────────────────────
print('\n=== Phase 2: Fine-tuning top 30 EfficientNetB0 layers ===')

backbone = model.layers[1]
backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.0),
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

callbacks_p2 = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, mode='max'),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, mode='max'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-8),
]

history_p2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=callbacks_p2,
)
print(f'Phase 2 best val_accuracy: {max(history_p2.history["val_accuracy"]):.4f}')

## 📊 Step 5 — Evaluate on Test Set

In [ ]:
print('\n=== Final Evaluation on Test Set ===')
results = model.evaluate(test_data, verbose=1)
metric_names = ['loss', 'accuracy', 'precision', 'recall']
for name, val in zip(metric_names, results):
    print(f'  {name}: {val:.4f}')

## 💾 Step 6 — Download Model to Your PC

In [ ]:
# Option A: Download directly from Colab
from google.colab import files
files.download(MODEL_SAVE_PATH)
print('✅ Download started. Place deepfake_detector_model.keras in your TruthLens/ folder.')

---
## ✅ Done!
Place the downloaded `deepfake_detector_model.keras` in your local `TruthLens/` folder, then run:
```bash
python inference.py
```